# 📥 06. ANFIS Data Preparation

### 🎯 Objective
Construct the final 6-feature dataset required for the Adaptive Network-Based Fuzzy Inference System (ANFIS).

### 🔗 Connection to Previous Step
This notebook consumes `5_clustered_telemetry.csv`, which was generated in **Step 05 (Clustering)**. 
It takes the **Soft Membership** scores (Combat/Collect/Explore) calculated via K-Means and combines them with **Temporal Deltas** to form a complete state vector for the AI controller.


### 🧹 Output Artifacts:
- `../../data/processed/6_anfis_dataset.csv`


In [16]:
import pandas as pd

INPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'
OUTPUT_PATH = '../../data/processed/6_anfis_dataset.csv'

print("Libraries imported")

Libraries imported


In [17]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


## 💡 Feature Engineering: Temporal Dynamics

To enable predictive capability, the system requires context not just of the player's *current state*, but their *trajectory of change*.

**The Delta Input:**
We calculate the first-order derivative (velocity) of the soft membership scores:

$$ \Delta(t) = \text{Membership}(t) - \text{Membership}(t-1) $$

**Rationale:**
A player with a high Combat score (0.8) might be *engaging* (Positive Delta) or *disengaging* (Negative Delta). 
The Adaptive AI must respond differently to these two scenarios: ramping up difficulty for engagement vs. tapering off for disengagement.


## ⚖️ Design Decision: Target Variable Formulation

Establishing a robust "Ground Truth" target ($T$) for the neural network was a critical challenge. We compared two distinct formulations:

### ❌ Option A: Static State Heuristic (Discarded)
Initially, $T$ was calculated solely based on current state (e.g., $0.22 \cdot \text{Combat} + 0.18 \cdot \text{Collect} \dots$).
*   **The Problem (Variance Collapse):** Players often remain in a single state for extended periods. 
*   **Empirical Result:** This produced a target variable with extremely low variance ($\sigma \approx 0.011$).
*   **consequence:** The Neural Network failed to converge, collapsing to predicting the mean value ($T \approx 1.0$) for all inputs, as the signal was indistinguishable from noise.

### ✅ Option B: Temporal Delta Heuristic (Selected)
We introduced **Temporal Deltas** (rate of change) as the primary driver of the target calculation.
*   **The Solution:** Weighting deltas heavily (e.g., $\beta_{comb} = 0.55$) rewards the *action* of changing state rather than the state itself.
*   **Empirical Result:** Target variance increased by **5.5x** ($\sigma \approx 0.062$).
*   **Consequence:** This provided a strong, discernible gradient for backpropagation, enabling the ANFIS network to learn distinct adaptive behaviors for engaging vs. disengaging players.


## 📊 Canonical Target Formula (Mathematical Derivation)

The target multiplier $T$ is calculated as a linear combination of centered state variables ($S_c$), temporal deltas ($\Delta$), and a penalty term for failure ($P_d$).

$$ T = B + \sum_{i \in \{comb, col, exp\}} (\alpha_i \cdot S_{c,i}) + \sum_{j \in \{comb, col, exp\}} (\beta_j \cdot \Delta_j) - \gamma \cdot P_d $$

**Where:**
*   $B = 0.9$ (Base Difficulty Multiplier)
*   $S_{c,i}$: Centered Soft Membership (State - 0.5)
*   $\Delta_j$: Temporal Delta (Current - Previous)
*   $P_d$: Normalized Death Rate (0 to 1)

**Coefficients (Sensitivity):**
*   $\alpha$: State Weights $[0.22, 0.18, 0.15]$ (Lower sensitivity to static state)
*   $\beta$: Delta Weights $[0.55, 0.40, 0.35]$ (**High** sensitivity to behavior change)
*   $\gamma$: Penalty Weight $0.25$ (Reduces difficulty upon repeated failure)


In [18]:
# Select 6-feature input for ANFIS
input_features = [
    'soft_combat', 'soft_collect', 'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

print("ANFIS Input features (6):")
for i, f in enumerate(input_features, 1):
    print(f"{i}. {f}")

ANFIS Input features (6):
1. soft_combat
2. soft_collect
3. soft_explore
4. delta_combat
5. delta_collect
6. delta_explore


In [19]:
# OPTION B: Canonical Formula (FINAL v2.2)
# Target variance maximized subject to fuzzy-membership constraints
import numpy as np

# ---------------------------------------------------------
# 1. Center the Inputs
# Subtracting 0.5 centers the membership probabilities (range 0-1) 
# around zero, creating positive/negative bias signals.
# ---------------------------------------------------------
combat_c = df['soft_combat'] - 0.5
collect_c = df['soft_collect'] - 0.5
explore_c = df['soft_explore'] - 0.5

# ---------------------------------------------------------
# 2. Prepare Deltas
# Fill NaNs with 0 to handle the first frame of gameplay.
# ---------------------------------------------------------
delta_combat = df['delta_combat'].fillna(0)
delta_collect = df['delta_collect'].fillna(0)
delta_explore = df['delta_explore'].fillna(0)

# ---------------------------------------------------------
# 3. Death Rate Normalization
# Scales death count relative to the 95th percentile to prevent outliers
# from skewing the penalty term.
# ---------------------------------------------------------
deaths = df['death_count'].fillna(0)
death_rate = deaths / (deaths.quantile(0.95) + 1.0)
death_rate = death_rate.clip(0, 1)

# ---------------------------------------------------------
# 4. Calculate Target Multiplier
# This formula is the 'Teacher' for the Neural Network.
# Weights (e.g. 0.55 for delta_combat) determine the sensitivity of the AI.
# High weight on Deltas = AI reacts fast to behavior changes.
# ---------------------------------------------------------
df['target_multiplier'] = (
    0.9 
    + 0.22 * combat_c 
    + 0.18 * collect_c 
    + 0.15 * explore_c 
    + 0.55 * delta_combat  # High reaction to combat changes
    + 0.40 * delta_collect 
    + 0.35 * delta_explore 
    - 0.25 * death_rate    # Penalty for frequent deaths
)

# ---------------------------------------------------------
# 5. Safety Clamp
# Ensure the multiplier stays within playable bounds (0.6x to 1.4x difficulty).
# ---------------------------------------------------------
# I chose 0.6 as the minimum bound (Safety Floor).
# Even if the math says '0.2', I can't let the game become 80% easier or it becomes boring.
# 0.6 (40% easier) is the maximum help I'm willing to give purely for engagement reasons.
df['target_multiplier'] = np.clip(df['target_multiplier'], 0.6, 1.4)

# Validation stats (print distribution to verify spread)
t = df['target_multiplier']
print('='*60)
print('Option B Target Statistics (FINAL)')
print('='*60)
print(f'Min:  {t.min():.4f}')
print(f'Max:  {t.max():.4f}')
print(f'Mean: {t.mean():.4f}')
print(f'Std:  {t.std():.4f}')
print(f'Span: {t.max()-t.min():.4f}')
print('='*60)

Option B Target Statistics (FINAL)
Min:  0.4320
Max:  1.0225
Mean: 0.7970
Std:  0.0649
Span: 0.5905


### 🛠️ My Implementation Data Pipeline

**Important Note for the Live System:**
To make this work in the real game, I realized I don't need to send the entire user history from the client.

1.  **Client:** Sends only the *Current* telemetry snapshot.
2.  **Server (The Brain):** Keeps a small memory of the *Previous* state for that user.
3.  **Calculation:** The server computes `Current - Previous` to get the **Deltas** needed for my formula above.

This keeps the network traffic light while still letting me use these complex temporal features.

In [20]:
# Save
final_cols = ['userId', 'timestamp', 'cluster'] + input_features + ['target_multiplier']
anfis_df = df[final_cols].copy()
anfis_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Saved to ../../data/processed/6_anfis_dataset.csv
